In [1]:
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.summary_statistics import SummaryStatistics
import pyspark.sql.functions as f
import pyspark.sql.types as t
from pyspark.sql.functions import log10, row_number
from pyspark.sql.window import Window
from gentropy.method.sumstat_quality_controls import SummaryStatisticsQC

Loading BokehJS ...

In [2]:
path_gwas1 = "/Users/yt4/Projects/otg_etl/input_gwas/R10_M13_LOWBACKPAIN.parquet"
path_ld_index = "/Users/yt4/Projects/otg_etl/input_gwas/ld_index"
path_study_index = "/Users/yt4/Projects/otg_etl/input_gwas/study_index_finngen/"
path_v2g = "/Users/yt4/Projects/otg_etl/input_gwas/v2g"
path_l2g_model = "/Users/yt4/Projects/otg_etl/input_gwas/l2g_model"
path_pics_out = "/Users/yt4/Projects/otg_etl/input_gwas/picsed_study_locus"
path_studylocus="/Users/yt4/Projects/otg_etl/input_gwas/picsed_study_locus/"

In [3]:
session = Session()
gwas1 = SummaryStatistics.from_parquet(session, path_gwas1)
study_index_finngen = StudyIndex.from_parquet(session, path_study_index)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


24/06/24 14:31:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/06/24 14:31:32 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
SummaryStatisticsQC.sumstat_qc_beta_check(gwas1).show()

+--------------------+--------------------+
|             studyId|           mean_beta|
+--------------------+--------------------+
|FINNGEN_R10_M13_L...|-4.53735548277616...|
+--------------------+--------------------+



In [5]:
qc_c=SummaryStatisticsQC.sumstat_qc_pz_check(gwas1,limit=1000)
qc_c.show()

24/03/10 16:43:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/03/10 16:43:21 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/03/10 16:43:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/03/10 16:43:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/03/10 16:43:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
+--------------------+--------------------+--------------------+
|             studyId|        mean_diff_pz|          se_diff_pz|
+--------------------+--------------------+--------------------+
|FINNGEN_R10_M13_L...|-1.17952578564604...|3.642455624497142E-5|
+--------------------+--------------------+--------------------+



In [6]:
qc_c.toPandas()

24/03/10 16:43:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/03/10 16:43:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/03/10 16:43:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/03/10 16:43:47 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/03/10 16:43:48 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


,studyId,mean_diff_pz,se_diff_pz
0,FINNGEN_R10_M13_LOWBACKPAIN,-1.179526e-07,0.000036


In [7]:
qc_c=SummaryStatisticsQC.gc_lambda_check(gwas1,limit=100000)
qc_c.show()

+--------------------+------------------+
|             studyId|         gc_lambda|
+--------------------+------------------+
|FINNGEN_R10_M13_L...|1.1818343101085051|
+--------------------+------------------+



In [8]:
qc_c=SummaryStatisticsQC.number_of_snps(gwas1,pval_threhod=5e-8)
qc_c.show()

+--------------------+----------+--------------+
|             studyId|n_variants|n_variants_sig|
+--------------------+----------+--------------+
|FINNGEN_R10_M13_L...|  21304314|           971|
+--------------------+----------+--------------+



In [9]:
SummaryStatisticsQC.sumstat_n_eff_check(gwas1,limit=100000,n_total=100000,min_count=100).show()

+--------------------+-------------------+
|             studyId|               se_N|
+--------------------+-------------------+
|FINNGEN_R10_M13_L...|0.16301086954591998|
+--------------------+-------------------+



In [10]:
SummaryStatisticsQC.get_quality_control_metrics(gwas1,limit=10000).show()

24/03/10 16:45:12 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/03/10 16:45:12 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/03/10 16:45:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/03/10 16:45:14 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/03/10 16:45:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
24/03/10 16:45:16 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+--------------------+--------------------+--------------------+--------------------+------------------+-----------------+----------+--------------+
|             studyId|           mean_beta|        mean_diff_pz|          se_diff_pz|              se_N|        gc_lambda|n_variants|n_variants_sig|
+--------------------+--------------------+--------------------+--------------------+------------------+-----------------+----------+--------------+
|FINNGEN_R10_M13_L...|-4.53735548277616...|-1.32952074604442...|4.107130898213367E-5|0.1726649001838615|1.181691668944459|  21304314|           971|
+--------------------+--------------------+--------------------+--------------------+------------------+-----------------+----------+--------------+



In [4]:
gwas2=SummaryStatistics.from_parquet(session, "../tests/gentropy/data_samples/sumstats_sample/GCST005523_chr18.parquet")
SummaryStatisticsQC.number_of_snps(gwas2,pval_threhod=5e-8).show()

+----------+----------+--------------+
|   studyId|n_variants|n_variants_sig|
+----------+----------+--------------+
|GCST005523|      1669|            29|
+----------+----------+--------------+



In [5]:
gwas2=gwas2.sanity_filter()
SummaryStatisticsQC.number_of_snps(gwas2).show()

+----------+----------+--------------+
|   studyId|n_variants|n_variants_sig|
+----------+----------+--------------+
|GCST005523|      1663|            29|
+----------+----------+--------------+



In [6]:
QC=SummaryStatisticsQC.get_quality_control_metrics(gwas2,limit=10000)
QC.show()

+----------+--------------------+--------------------+--------------------+----+------------------+----------+--------------+
|   studyId|           mean_beta|        mean_diff_pz|          se_diff_pz|se_N|         gc_lambda|n_variants|n_variants_sig|
+----------+--------------------+--------------------+--------------------+----+------------------+----------+--------------+
|GCST005523|0.001310962803607386|7.436809245043267...|1.117746706241476...|null|1.9159734429793385|      1663|            29|
+----------+--------------------+--------------------+--------------------+----+------------------+----------+--------------+



In [14]:
from pyspark.sql import functions as f

#QC=SummaryStatisticsQC.get_quality_control_metrics(XXX)

# Define the conditions for the new column
conditions = (
    (f.abs(f.col("mean_beta")) <= 0.05) &
    (f.abs(f.col("mean_diff_pz")) <= 0.05) &
    (f.abs(f.col("se_diff_pz")) <= 0.05) &
    (f.col("gc_lambda") <= 2) &
    (f.col("n_variants") >= 1_000_000) &
    ((f.col("se_N") <= 2) | f.col("se_N").isNull())
)

# Add the new column
QC = QC.withColumn("passedQC", f.when(conditions, True).otherwise(False))

In [13]:
QC.show()

+----------+--------------------+--------------------+--------------------+----+------------------+----------+--------------+--------+
|   studyId|           mean_beta|        mean_diff_pz|          se_diff_pz|se_N|         gc_lambda|n_variants|n_variants_sig|passedQC|
+----------+--------------------+--------------------+--------------------+----+------------------+----------+--------------+--------+
|GCST005523|0.001310962803607386|7.436809245043267...|1.117746706241476...|null|1.9159734429793385|      1663|            29|   false|
+----------+--------------------+--------------------+--------------------+----+------------------+----------+--------------+--------+



In [32]:
QC.toPandas()

,studyId,mean_beta,mean_diff_pz,se_diff_pz,se_N,gc_lambda,n_variants,n_variants_sig
0,GCST005523,0.001311,7.436809e-11,1.117747e-08,NaN,1.915973,1663,29


In [14]:
SummaryStatisticsQC.gc_lambda_check(gwas2).show()

+----------+------------------+
|   studyId|         gc_lambda|
+----------+------------------+
|GCST005523|1.9159734429793385|
+----------+------------------+



In [15]:
SummaryStatisticsQC.sumstat_qc_beta_check(gwas2).show()

+----------+--------------------+
|   studyId|           mean_beta|
+----------+--------------------+
|GCST005523|0.001310962803607386|
+----------+--------------------+



In [16]:
SummaryStatisticsQC.sumstat_qc_pz_check(gwas2).show()

+----------+--------------------+--------------------+
|   studyId|        mean_diff_pz|          se_diff_pz|
+----------+--------------------+--------------------+
|GCST005523|7.436809245043267...|1.117746706241476...|
+----------+--------------------+--------------------+



In [17]:
SummaryStatisticsQC.sumstat_n_eff_check(gwas2,limit=1000,n_total=100000,min_count=100).show()

+-------+----+
|studyId|se_N|
+-------+----+
+-------+----+



In [18]:
gwas_df=gwas2._df
gwas_df=gwas_df.withColumn('effectAlleleFrequencyFromSource', f.lit(0.5))
gwas2._df=gwas_df

In [19]:
SummaryStatisticsQC.sumstat_n_eff_check(gwas2,limit=100000,n_total=100000,min_count=100).show()

+----------+------------------+
|   studyId|              se_N|
+----------+------------------+
|GCST005523|0.5585536469698386|
+----------+------------------+



In [27]:
from pyspark.sql.functions import when, rand

gwas=gwas2
gwas_df = gwas._df
gwas_df = gwas_df.withColumn('studyId', when(rand() < 0.5, 'new_value').otherwise(gwas_df['studyId']))
gwas._df = gwas_df

QC = SummaryStatisticsQC.get_quality_control_metrics(
    gwas=gwas, limit=100000, min_count=100, n_total=100000
)
QC = QC.toPandas()

In [28]:
QC

,studyId,mean_beta,mean_diff_pz,se_diff_pz,se_N,gc_lambda,n_variants,n_variants_sig
0,GCST005523,0.003332,1.731177e-10,1.130897e-08,0.560712,1.915973,817,12
1,new_value,-0.000641,-2.099643e-11,1.105484e-08,0.558211,1.905659,846,17


In [ ]:
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.summary_statistics import SummaryStatistics
import pyspark.sql.functions as f
import pyspark.sql.types as t
from pyspark.sql.functions import log10, row_number
from pyspark.sql.window import Window
from gentropy.method.sumstat_quality_controls import SummaryStatisticsQC


fg_list=["gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_ACTINOMYCOSIS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_AMOEBIASIS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_ANOGENITAL_HERPES_SIMPLEX.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_ASPERGILLOSIS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_ATYPICAL_CNS_VIRUS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_BACTINF_NOS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_BACT_BIR_OTHER_INF_AGENTS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_BACT_INTEST_OTH.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_CANDIDIASIS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_CHLAMYDIA_OTHER.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_CHOLERA.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_DENGUE_FEVER.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_DERMATOPHYTOSIS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_EARLY_SYPHILIS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_EBV.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_ENTEROBIASIS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_ERYSIPELAS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_GASTROENTERITIS_NOS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_GONOCOCCAL.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_HELMINTIASES.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_HEPATITIS_A.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_HEPATITIS_B.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_HERPES_SIMPLEX.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_HIV_INFECT_PARASITES.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_HIV_NOS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_INTESTINAL_INFECTIONS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_LATE_SYPHILIS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_LISTERIOSIS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_LYME.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_MALARIA.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_MALARIA_FALCIPARIUM.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_MALARIA_NOS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_MEASLES.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_MENINGOCOCCAL.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_MUMPS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_MYCOSES.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_MYSOSIS_NOS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_OTHER_BACTERIAL.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_OTHER_BACT_INOTHER.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_OTHER_CESTODE.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_OTHER_CHLAMYDIAE.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_OTHER_CONDITIONS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_OTHER_INFECTIONS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_OTHER_INFESTATIONS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_OTHER_MOSQUITO_FEVER.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_OTHER_MYCOBAC.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_OTHER_PROTOZOAL.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_OTHER_SEPSIS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_OTHER_SPIROCHAETAL.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_OTHER_SUPERF_MYCOSIS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_OTHER_VIRAL.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_PARASITIC_NOS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_PEDICULOSIS_ACARIASIS_OTHERINFEST.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_PEDICULOSIS_AND_PHTHIRIAASIS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_PNEUMOCYSTOSIS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_POLIOMYELITIS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_PROTOZOAL.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_RELAP_FEVER.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_RESP_TUBERCU_CONF.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_RESP_TUBERCU_NONCONF.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_RUBELLA.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_SALMONELLA_OTH.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_SCABIES.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_SCARLET_FEVER.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_SEQULAE_INF_NOS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_SEQULAE_POLIO.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_SEQULAE_TUBERCU.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_SEQULEAE_OF_INFECTIONS_AND_PARASITES.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_SEXUAL_TRANSMISSION.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_SEXUAL_TRANSMISSION_NOS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_SHIGELLOSIS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_STREPTO_SEPSIS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_STREPTSTAPHYCOCC_INOTHER.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_SYPHILIS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_SYPHILIS_NOS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_TICK_BORNE_ENCEPHALITIS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_TOXOPLASMOSIS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_TRICHOMONIASIS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_TUBERCULOSIS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_TUBERCU_OTH_ORG.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_TULARAEMIA.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_TYPHOIDPARATHYPHOID.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_VARICELLA.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_VIRAAL_CONJUNCT.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_VIRAL_CNS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_VIRAL_ENCEPHALITIS_NOS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_VIRAL_HEMOR_FEVER_NOS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_VIRAL_HEPATITIS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_VIRAL_INOTHER.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_VIRAL_MENINGITIS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_VIRAL_NOS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_VIRAL_OTHER_INTEST_INFECTIONS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_VIRAL_SKIN_MUCOUS_MEMBRANE.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_VIRAL_SKIN_NOS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_VIRAL_WARTS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_VIRAN_CNS_INFECTIONS_NOS.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_WHOOPCOUGH.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_ZOONOTIC_BACTERIAL.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_AB1_ZOSTER.parquet",
"gs://finngen_data/r10/harmonised_summary_statistics/R10_ABDOM_HERNIA.parquet"]

session = Session()
gwas_fg = SummaryStatistics.from_parquet(session, fg_list)

In [ ]:
QC=SummaryStatisticsQC.get_quality_control_metrics(gwas_fg,limit=10000)
QC.show()